# MVP, Предсказание расхода электроэнергии для станции

## 1.1 Подготовка данных

In [174]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
import pytorch_forecasting as pf

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

warnings.filterwarnings("ignore")

`Фиксация SEED для воспроизводимости`

In [175]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

`Настройка глобальных переменных`

In [176]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET = "Выработка"
GROUP = "station_id"

ENCODER_LENGTH = 224
PREDICT_LENGTH = 24
BATCH_SIZE = 128

TRAIN_PATH = "data/dataset/train_dataset.csv"
VALID_PATH = "data/dataset/valid_features.csv"

`Загрузка датасетов`

In [177]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

## 1.2 Базовая предобработка

In [178]:
train_df.columns

Index(['METEOFORECASTHOUR_OPENM_Datetime', 'month', 'hour_of_day',
       'Выработка. Результирующий расчет', 'wind_speed_10m', 'wind_speed_80m',
       'wind_speed_120m', 'wind_speed_180m', 'wind_direction_10m',
       'wind_direction_80m', 'wind_direction_120m', 'wind_direction_180m',
       'wind_gusts_10m', 'temperature_80m', 'temperature_120m', 'pressure_msl',
       'rain', 'showers', 'snowfall', 'cloud_cover_low',
       'Кол-во_ВЭУ_в_ремонте'],
      dtype='object')

In [179]:
train_df.describe()

,month,hour_of_day,Выработка. Результирующий расчет,wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_speed_180m,wind_direction_10m,wind_direction_80m,wind_direction_120m,wind_direction_180m,wind_gusts_10m,temperature_80m,temperature_120m,pressure_msl,rain,showers,snowfall,cloud_cover_low,Кол-во_ВЭУ_в_ремонте
count,32434.000000,32434.000000,32434.000000,32434.00000,32434.000000,32434.000000,25598.000000,32434.000000,32434.000000,32434.000000,25598.000000,32434.000000,32434.000000,32434.000000,32434.000000,32434.000000,32434.000000,32434.000000,32434.00000,32434.000000
mean,6.530709,11.529259,33.005746,4.23351,6.549474,7.420251,8.195232,0.166194,0.167781,0.169408,0.168660,7.261503,12.264694,12.049941,1016.369853,0.032589,0.015456,0.002398,0.03085,4.021798
std,3.483929,6.929682,26.121408,2.11209,2.917924,3.346307,4.083580,0.102614,0.102019,0.101581,0.099701,3.581453,10.051639,10.114166,7.405932,0.211354,0.123338,0.030991,0.03818,0.983375
min,1.000000,0.000000,0.001000,0.10000,0.000000,0.100000,0.100000,0.001000,0.001000,0.001000,0.000000,0.200000,-15.300000,-16.000000,987.300000,0.000000,0.000000,0.000000,-0.00100,3.000000
25%,3.000000,6.000000,8.838750,2.69000,4.390000,4.900000,4.990000,0.074000,0.077000,0.080000,0.083000,4.600000,3.900000,3.700000,1011.300000,0.000000,0.000000,0.000000,0.00000,3.000000
50%,7.000000,12.000000,27.124500,3.92000,6.300000,7.120000,7.660000,0.156000,0.156000,0.157000,0.149000,6.800000,11.900000,11.800000,1015.600000,0.000000,0.000000,0.000000,0.00500,4.000000
75%,10.000000,18.000000,56.943250,5.51000,8.347500,9.690000,10.980000,0.256000,0.259000,0.260000,0.258000,9.400000,20.900000,20.600000,1021.100000,0.000000,0.000000,0.000000,0.06600,5.000000
max,12.000000,23.000000,87.475000,14.91000,20.610000,21.990000,25.110000,0.360000,0.360000,0.360000,0.360000,27.900000,37.200000,36.900000,1042.100000,7.500000,5.500000,1.050000,0.10100,7.000000


In [180]:
train_df.isna().sum()

METEOFORECASTHOUR_OPENM_Datetime       0
month                                  0
hour_of_day                            0
Выработка. Результирующий расчет       0
wind_speed_10m                         0
wind_speed_80m                         0
wind_speed_120m                        0
wind_speed_180m                     6836
wind_direction_10m                     0
wind_direction_80m                     0
wind_direction_120m                    0
wind_direction_180m                 6836
wind_gusts_10m                         0
temperature_80m                        0
temperature_120m                       0
pressure_msl                           0
rain                                   0
showers                                0
snowfall                               0
cloud_cover_low                        0
Кол-во_ВЭУ_в_ремонте                   0
dtype: int64

`Высота ротора 80м, информация о направлении и скорости ветра на высоте 180м будет слабо влиять на результат`

In [181]:
train_df.drop(columns=['wind_speed_180m', 'wind_direction_180m'], axis=1, inplace=True, errors='ignore')  # Удаление слабо влияющей колонки
valid_df.drop(columns=['wind_speed_180m', 'wind_direction_180m'], axis=1, inplace=True, errors='ignore')  # Удаление слабо влияющей колонки

In [182]:
train_df.duplicated().sum()

np.int64(0)

`Переименование target-column с целью предотвращения ошибок`

In [183]:
train_df.rename({"Выработка. Результирующий расчет": "Выработка"}, axis=1, inplace=True, errors='ignore')
valid_df.rename({"Выработка. Результирующий расчет": "Выработка"}, axis=1, inplace=True, errors='ignore')

`Приведение timestamp к datetime-типу`

In [184]:
train_df["METEOFORECASTHOUR_OPENM_Datetime"] = pd.to_datetime(train_df["METEOFORECASTHOUR_OPENM_Datetime"])
valid_df["METEOFORECASTHOUR_OPENM_Datetime"] = pd.to_datetime(valid_df["METEOFORECASTHOUR_OPENM_Datetime"])

train_df = train_df.sort_values("METEOFORECASTHOUR_OPENM_Datetime").reset_index(drop=True)
valid_df = valid_df.sort_values("METEOFORECASTHOUR_OPENM_Datetime").reset_index(drop=True)

`Заглушка для TFS-модели`

In [185]:
train_df[GROUP] = "station_1"
valid_df[GROUP] = "station_1"

## 1.3 Preprocessing & Feature Engineering

In [186]:
train_df["time_idx"] = np.arange(len(train_df))
last_idx = train_df["time_idx"].max()

In [187]:
valid_df["time_idx"] = np.arange(
    last_idx + 1,
    last_idx + 1 + len(valid_df)
)

In [188]:
for df in [train_df, valid_df]:

    num_cols = df.select_dtypes(
        include=["float64", "float32", "int64", "int32"]
    ).columns

    for col in num_cols:

        if col != "time_idx":
            df[col] = df[col].replace(
                [np.inf, -np.inf],
                np.nan
            )

In [189]:
weather_cols = [
    c for c in valid_df.columns
    if c not in [
        TARGET,
        GROUP,
        "METEOFORECASTHOUR_OPENM_Datetime",
        "time_idx"
    ]
]

In [190]:
train_df[weather_cols] = train_df[weather_cols].ffill()
valid_df[weather_cols] = valid_df[weather_cols].ffill()

In [191]:
for lag in [24, 48, 72, 168]:
    train_df[f"lag_{lag}"] = train_df[TARGET].shift(lag)

In [192]:
for roll in [24, 48, 168]:
    train_df[f"roll_mean_{roll}"] = (
        train_df[TARGET]
        .rolling(roll)
        .mean()
    )

In [193]:
train_df[f"roll_std_{roll}"] = (
        train_df[TARGET]
        .rolling(roll)
        .std()
    )

In [194]:
if "wind_speed" in train_df.columns:
    train_df["wind_speed_cube"] = train_df["wind_speed"] ** 3
    valid_df["wind_speed_cube"] = valid_df["wind_speed"] ** 3

In [195]:
train_df = train_df.ffill().dropna().reset_index(drop=True)

## 1.3 Подготовка данных к обучению

In [196]:
known_reals = [
    c for c in valid_df.columns
    if c not in [TARGET, GROUP]
    and c != "timestamp"
]

In [197]:
unknown_reals = [
    TARGET,
    *[c for c in train_df.columns if c.startswith("lag_")],
    *[c for c in train_df.columns if c.startswith("roll_")]
]

In [198]:
training_cutoff = train_df["time_idx"].max() - 2160

In [199]:
train_dataset = TimeSeriesDataSet(
    train_df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target=TARGET,
    group_ids=[GROUP],
    min_encoder_length=ENCODER_LENGTH,
    max_encoder_length=ENCODER_LENGTH,
    min_prediction_length=PREDICT_LENGTH,
    max_prediction_length=PREDICT_LENGTH,
    static_categoricals=[GROUP],
    time_varying_known_reals=known_reals,
    time_varying_unknown_reals=unknown_reals,
    target_normalizer=GroupNormalizer(
        groups=[GROUP],
        transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

In [206]:
torch.save(train_dataset, "train_dataset.pth")

In [200]:
val_dataset = TimeSeriesDataSet.from_dataset(
    train_dataset,
    train_df,
    min_prediction_idx=training_cutoff + 1,
    stop_randomization=True
)

In [201]:
train_loader = train_dataset.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0
)

val_loader = val_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

## 2.1 Настройка обучения

In [202]:
trainer = pl.Trainer(
    accelerator=DEVICE,
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            mode="min"
        ),
        LearningRateMonitor()
    ],
    logger=TensorBoardLogger("logs"),
    enable_checkpointing=True
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [203]:
TFT = pf.TemporalFusionTransformer.from_dataset(
    train_dataset,
    learning_rate=1e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.2,
    hidden_continuous_size=32,
    loss=QuantileLoss(),
    reduce_on_plateau_patience=3
)

## 2.2 Обучение TFT-модели

In [204]:
trainer.fit(
    TFT,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      1 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  2.0 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 20.8 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  226 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  149 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 603 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 603 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 926                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [205]:
train_df.to_csv("train_dataset.csv", index=False)

## 2.3 Inference

`Inference`

In [207]:
best_model_path = trainer.checkpoint_callback.best_model_path

model = pf.TemporalFusionTransformer.load_from_checkpoint(best_model_path)
model.to(DEVICE)
model.eval()

TemporalFusionTransformer(
  	"attention_head_size":               4
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'Выработка', 'group_ids': ['station_id'], 'weight': None, 'max_encoder_length': 224, 'min_encoder_length': 224, 'min_prediction_idx': 168, 'min_prediction_length': 24, 'max_prediction_length': 24, 'static_categoricals': ['station_id'], 'static_reals': None, 'time_varying_known_categoricals': None, 'time_varying_known_reals': ['METEOFORECASTHOUR_OPENM_Datetime', 'month', 'hour_of_day', 'wind_speed_10m', 'wind_speed_80m', 'wind_speed_120m', 'wind_direction_10m', 'wind_direction_80m', 'wind_direction_120m', 'wind_gusts_10m', 'temperature_80m', 'temperature_120m', 'pressure_msl', 'rain', 'showers', 'snowfall', 'cloud_cover_low', 'Кол-во_ВЭУ_в_ремонте', 'time_idx'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['Выработка', 'lag_24', '

In [208]:
quantiles = model.loss.quantiles
median_idx = quantiles.index(0.5)

In [209]:
valid_df[TARGET] = np.nan

In [210]:
for lag in [24, 48, 72, 168]:
    valid_df[f"lag_{lag}"] = np.nan

for roll in [24, 48, 168]:
    valid_df[f"roll_mean_{roll}"] = np.nan
    valid_df[f"roll_std_{roll}"] = np.nan

In [211]:
history = train_df.tail(ENCODER_LENGTH).copy()

predictions = []

for start in range(0, len(valid_df), PREDICT_LENGTH):

    end = min(
        start + PREDICT_LENGTH,
        len(valid_df)
    )

    chunk = valid_df.iloc[start:end].copy()

    real_decoder_len = len(chunk)

    chunk[TARGET] = np.nan

    temp = pd.concat(
        [history, chunk],
        ignore_index=True
    )

    temp = temp.sort_values(
        "time_idx"
    ).reset_index(drop=True)

    temp[TARGET] = temp[TARGET].ffill()

    for lag in [24, 48, 72, 168]:

        temp[f"lag_{lag}"] = (
            temp[TARGET]
            .shift(lag)
        )

    for roll in [24, 48, 168]:

        temp[f"roll_mean_{roll}"] = (
            temp[TARGET]
            .rolling(
                roll,
                min_periods=1
            )
            .mean()
        )

        temp[f"roll_std_{roll}"] = (
            temp[TARGET]
            .rolling(
                roll,
                min_periods=1
            )
            .std()
        )

    feature_cols = [
        c for c in temp.columns
        if c not in [
            "METEOFORECASTHOUR_OPENM_Datetime"
        ]
    ]

    temp[feature_cols] = (
        temp[feature_cols]
        .ffill()
        .bfill()
    )

    pred_dataset = TimeSeriesDataSet.from_dataset(
        train_dataset,
        temp,
        stop_randomization=True,
        predict=False,
        min_prediction_length=real_decoder_len,
        max_prediction_length=real_decoder_len
    )

    pred_loader = pred_dataset.to_dataloader(
        train=False,
        batch_size=1,
        num_workers=0
    )

    raw_preds = model.predict(
        pred_loader,
        mode="raw",
        return_x=False
    )

    pred = raw_preds.prediction.detach().cpu().numpy()

    if pred.ndim == 3:
        pred = pred[0, :, median_idx]
    elif pred.ndim == 2:
        pred = pred[0]

    pred = pred[:real_decoder_len]

    chunk[TARGET] = pred

    predictions.extend(pred.tolist())

    history = pd.concat(
        [history, chunk],
        ignore_index=True
    ).tail(ENCODER_LENGTH)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelC

In [212]:
predictions = np.array(predictions)

predictions = np.clip(predictions, 0, None)

assert len(predictions) == len(valid_df)

submission = pd.DataFrame(predictions)

submission.to_csv(
    "submission.csv",
    index=False,
    header=False
)

print(submission.shape)
print(submission.head())

(2126, 1)
           0
0  47.477615
1  38.976028
2  43.853706
3  47.279419
4  52.738647


In [42]:
predictions

array([40.20956039, 36.530056  , 40.6366272 , ..., 20.37553978,
       23.83005142, 25.93103027], shape=(2126,))

In [43]:
len(valid_df)

2126

In [215]:
!pip list --format=freeze > requirements.txt